# Training a CNN on MNIST with PyTorch Lightning

This notebook demonstrates how to train a simple, efficient Convolutional Neural Network (CNN) with residual connections on the MNIST dataset using PyTorch Lightning. The configuration is tuned for ≥99.7% accuracy.

In [1]:
!pip install pytorch-lightning torch torchvision datasets wandb pillow scikit-learn

In [2]:
from PIL import ExifTags, Image
Image.ExifTags = ExifTags  # Hack to bypass broken import

Define config:

In [3]:
import os

def setup_config():
    # General Settings
    model_id = "resnet18"#"resnet50"
    backbone_warmup_percentage = 0.1
    dataset = "cifar10"  # Options: "mnist" or "cifar10"
    seed = 42
    n_epochs = 100
    batch_size = 256
    # Optimizer Settings
    learning_rate = 1e-3
    weight_decay = 0
    warmup_ratio = 0
    max_grad_norm = 0
    # Model Architecture
    nonlinearity = 'relu'
    weight_init = 'kaiming'
    # Data Settings
    sequence_length = 1
    train_size = 0.8
    val_size = 0.2
    # Softcoded new params
    label_smoothing = 0.0
    dropout = 0.1
    early_stopping_patience = 7
    early_stopping_min_delta = 1e-4
    use_mixed_precision = True
    swa_lrs = 0#1e-2
    activation = "relu"
    

    transform_affine=False
    transform_rotation=False
    transform_erasing=False
    transform_crop=True
    transform_hflip=True
    transform_vflip=False
    transform_color_jitter=True

    os.environ["NOTEBOOK_ID"] = "mnist-cnn-pl"

    return {
        'model_id': model_id,
        'backbone_warmup_percentage' : backbone_warmup_percentage,
        'dataset': dataset,
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'sequence_length': sequence_length,
        'nonlinearity': nonlinearity,
        'weight_init': weight_init,
        'weight_decay': weight_decay,
        'warmup_ratio': warmup_ratio,
        'max_grad_norm': max_grad_norm,
        'train_size': train_size,
        'val_size': val_size,
        'label_smoothing': label_smoothing,
        'dropout': dropout,
        'early_stopping_patience': early_stopping_patience,
        'early_stopping_min_delta': early_stopping_min_delta,
        'use_mixed_precision': use_mixed_precision,
        'swa_lrs': swa_lrs,
        'activation': activation,
        'transform_affine': transform_affine,
        'transform_rotation': transform_rotation,
        'transform_erasing': transform_erasing,
        'transform_crop': transform_crop,
        'transform_hflip': transform_hflip,
        'transform_vflip': transform_vflip,
        'transform_color_jitter': transform_color_jitter
    }

CONFIG = setup_config()

Set seed for reproducibility:

In [4]:
import pytorch_lightning as pl
pl.seed_everything(CONFIG['seed'], workers=True)

Seed set to 42


42

Set matmul precision:

In [5]:
import torch
torch.set_float32_matmul_precision('high')

Login to wandb:

In [6]:
import wandb
wandb.login()

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
from torchvision import transforms

# Dataset specifications with only essential info
DATASET_SPECS = {
    "imagenet": {
        "image_size": 224,
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "cifar10": {
        "image_size": 32,
        "mean": [0.4914, 0.4822, 0.4465],
        "std": [0.2023, 0.1994, 0.2010]
    },
    "mnist": {
        "image_size": 28,
        "mean": [0.1307],
        "std": [0.3081]
    }
}

def build_dataset_transforms(dataset_id: str, config: dict):
    assert dataset_id in DATASET_SPECS, f"Unknown dataset spec: {dataset_id}"
    spec = DATASET_SPECS[dataset_id]

    image_size = spec["image_size"]
    crop_fraction = 0.875
    resize_size = int(round(image_size / crop_fraction))

    preprocessing_pipeline = [
        transforms.Resize(resize_size),
        transforms.CenterCrop(image_size)
    ]

    aug = []
    if config.get("transform_affine"): aug.append(transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)))
    if config.get("transform_rotation"): aug.append(transforms.RandomRotation(degrees=15))
    if config.get("transform_crop"): aug.append(transforms.RandomResizedCrop(image_size))
    if config.get("transform_hflip"): aug.append(transforms.RandomHorizontalFlip())
    if config.get("transform_vflip"): aug.append(transforms.RandomVerticalFlip())
    if config.get("transform_color_jitter"): aug.append(transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1))
    augmentation_pipeline = aug

    normalization_pipeline = [
        transforms.ToTensor(),
        transforms.Normalize(mean=spec["mean"], std=spec["std"]),
    ]

    train_transform = transforms.Compose(
        preprocessing_pipeline + augmentation_pipeline + normalization_pipeline
    )
    test_transform = transforms.Compose(
        preprocessing_pipeline + normalization_pipeline
    )

    return train_transform, test_transform


Define MNIST data module:

In [8]:
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader

class MNISTDataModule(pl.LightningDataModule):
    def __init__(
        self, 
        batch_size, 
        train_size, 
        seed, 
        num_workers=2, 
        train_shuffle=True,
        val_shuffle=False,
        test_shuffle=False,
        transform_affine=False, 
        transform_rotation=False, 
        transform_erasing=False,
        transform_crop=False,
        transform_hflip=False,
        transform_vflip=False,
        transform_color_jitter=False
    ):
        super().__init__()
        self.dataset_id = "mnist"
        self.download_path = f"./temp/{self.dataset_id}"
        self.batch_size = batch_size
        self.train_size = train_size
        self.seed = seed
        self.num_workers = num_workers
        self.train_shuffle = train_shuffle
        self.val_shuffle = val_shuffle
        self.test_shuffle = test_shuffle
        self.n_classes = 10
        self.transform_affine = transform_affine
        self.transform_rotation = transform_rotation
        self.transform_erasing = transform_erasing
        self.transform_crop = transform_crop
        self.transform_hflip = transform_hflip
        self.transform_vflip = transform_vflip
        self.transform_color_jitter = transform_color_jitter


    def prepare_data(self):
        MNIST(root=self.download_path, train=True, download=True)
        MNIST(root=self.download_path, train=False, download=True)

    def setup(self, stage=None):
        self.train_transform, self.test_transform = build_dataset_transforms(self.dataset_id, {
            "transform_affine": self.transform_affine,
            "transform_rotation": self.transform_rotation,
            "transform_erasing": self.transform_erasing,
            "transform_crop": self.transform_crop,
            "transform_hflip": self.transform_hflip,
            "transform_vflip": self.transform_vflip,
            "transform_color_jitter": self.transform_color_jitter
        })

        """
        transform_list = []
        if self.transform_affine: transform_list.append(transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)))
        if self.transform_rotation: transform_list.append(transforms.RandomRotation(degrees=15))
        transform_list.append(transforms.ToTensor())
        transform_list.append(transforms.Normalize((0.1307,), (0.3081,)))
        if self.transform_erasing: transform_list.append(transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random'))

        self.train_transform = transforms.Compose(transform_list)
        self.test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        """

        full = MNIST(root=self.download_path, train=True, transform=self.train_transform)
        total = len(full)
        train_size = int(total * self.train_size)
        val_size = total - train_size
        self.train_set, self.val_set = torch.utils.data.random_split(
            full, [train_size, val_size], generator=torch.Generator().manual_seed(self.seed)
        )
        self.val_set.dataset.transform = self.test_transform
        self.test_set = MNIST(root=self.download_path, train=False, transform=self.test_transform)

    def train_dataloader(self):
        return DataLoader(
            self.train_set, 
            batch_size=self.batch_size, 
            shuffle=self.train_shuffle, 
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_set, 
            batch_size=self.batch_size, 
            shuffle=self.val_shuffle, 
            num_workers=self.num_workers
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_set, 
            batch_size=self.batch_size, 
            shuffle=self.test_shuffle, 
            num_workers=self.num_workers
        )

Define CIFAR10 data module:

In [9]:
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

class CIFAR10DataModule(pl.LightningDataModule):
    def __init__(
        self, 
        batch_size, 
        train_size, 
        seed, 
        num_workers=2, 
        train_shuffle=True,
        val_shuffle=False,
        test_shuffle=False,
        transform_affine=False, 
        transform_rotation=False, 
        transform_erasing=False,
        transform_crop=False,
        transform_hflip=False,
        transform_vflip=False,
        transform_color_jitter=False
    ):
        super().__init__()
        self.dataset_id = "cifar10"
        self.download_path = f"./temp/{self.dataset_id}"
        self.batch_size = batch_size
        self.train_size = train_size
        self.seed = seed
        self.num_workers = num_workers
        self.train_shuffle = train_shuffle
        self.val_shuffle = val_shuffle
        self.test_shuffle = test_shuffle
        self.n_classes = 10

        # TODO: softcode transforms
        self.transform_affine = transform_affine
        self.transform_rotation = transform_rotation
        self.transform_erasing = transform_erasing
        self.transform_crop = transform_crop
        self.transform_hflip = transform_hflip
        self.transform_vflip = transform_vflip
        self.transform_color_jitter = transform_color_jitter

    def prepare_data(self):
        CIFAR10(root=self.download_path, train=True, download=True)
        CIFAR10(root=self.download_path, train=False, download=True)

    def setup(self, stage=None):
        self.train_transform, self.test_transform = build_dataset_transforms("imagenet", {
            "transform_affine": self.transform_affine,
            "transform_rotation": self.transform_rotation,
            "transform_erasing": self.transform_erasing,
            "transform_crop": self.transform_crop,
            "transform_hflip": self.transform_hflip,
            "transform_vflip": self.transform_vflip,
            "transform_color_jitter": self.transform_color_jitter
        })


        """
        transform_list = []
        if self.transform_crop: transform_list.append(transforms.RandomCrop(32, padding=4))
        if self.transform_hflip: transform_list.append(transforms.RandomHorizontalFlip())
        if self.transform_color_jitter: transform_list.append(transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1))
        transform_list.append(transforms.ToTensor())
        transform_list.append(transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)))

        self.train_transform = transforms.Compose(transform_list)
        self.test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])

        # TODO: SOFTCODE transform pipeline
        # Training transforms
        imagenet_mean = [0.485, 0.456, 0.406]
        imagenet_std = [0.229, 0.224, 0.225]
        self.train_transform = transforms.Compose([
            transforms.Resize(256),                      # Resize to larger scale before crop
            transforms.RandomResizedCrop(224),           # Random crop to 224x224
            transforms.RandomHorizontalFlip(),           # Data augmentation
            transforms.ToTensor(),
            transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
        ])

        # Validation/test transforms
        self.test_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
        ])
        """

        full = CIFAR10(root=self.download_path, train=True, transform=self.train_transform)
        total = len(full)
        train_size = int(total * self.train_size)
        val_size = total - train_size
        self.train_set, self.val_set = torch.utils.data.random_split(
            full, [train_size, val_size], generator=torch.Generator().manual_seed(self.seed)
        )
        self.val_set.dataset.transform = self.test_transform
        self.test_set = CIFAR10(root=self.download_path, train=False, transform=self.test_transform)

    def train_dataloader(self):
        return DataLoader(
            self.train_set, 
            batch_size=self.batch_size, 
            shuffle=self.train_shuffle, 
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_set, 
            batch_size=self.batch_size, 
            shuffle=self.val_shuffle, 
            num_workers=self.num_workers
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_set, 
            batch_size=self.batch_size, 
            shuffle=self.test_shuffle, 
            num_workers=self.num_workers
        )



Create the data module:

In [10]:
def create_data_module(dataset_id, **kwargs):
    defaults = {
        "batch_size": CONFIG['batch_size'],
        "train_size": CONFIG['train_size'],
        "seed": CONFIG['seed'],
        "transform_affine": CONFIG['transform_affine'],
        "transform_rotation": CONFIG['transform_rotation'],
        "transform_erasing": CONFIG['transform_erasing'],
        "transform_crop":  CONFIG['transform_crop'],
        "transform_hflip": CONFIG['transform_hflip'],
        "transform_vflip": CONFIG['transform_vflip'],
        "transform_color_jitter": CONFIG['transform_color_jitter']
    }
    params = {**defaults, **kwargs}

    dataset_modules = {
        "mnist": MNISTDataModule,
        "cifar10": CIFAR10DataModule
    }
    dataset_id = dataset_id.lower()
    datamodule_class = dataset_modules.get(dataset_id)
    assert datamodule_class is not None, f"Unsupported dataset: {dataset_id}"
    datamodule = datamodule_class(**params)
    datamodule.prepare_data()
    return datamodule

dm = create_data_module(CONFIG['dataset'])

Files already downloaded and verified
Files already downloaded and verified


Create the model:

In [11]:
from torch import nn

class LitModel(pl.LightningModule):
    def __init__(self, lr=None, n_classes=None):
        super().__init__()
        self.save_hyperparameters()

        assert n_classes is not None, "n_classes must be provided"
        
        self.model = torch.hub.load("pytorch/vision", CONFIG["model_id"], weights="DEFAULT")
        
        self.model.fc = nn.Linear(self.model.fc.in_features, n_classes)

        self.criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
    
    def freeze_backbone(self):
        for param in self.model.parameters(): param.requires_grad = False
        for param in self.model.fc.parameters(): param.requires_grad = True

    def unfreeze_backbone(self):
        for param in self.model.parameters(): param.requires_grad = True
        for param in self.model.fc.parameters(): param.requires_grad = True

    def forward(self, x):
        x = self.model(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train/loss", loss, prog_bar=True)
        self.log("train/acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        self.log("val/loss", loss, prog_bar=True, sync_dist=True)
        self.log("val/acc", acc, prog_bar=True, sync_dist=True)
        return {}

    def on_validation_epoch_end(self):
        pass

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test/loss", loss)
        self.log("test/acc", acc)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            # TODO: why not only self.hparams.lr
            lr=self.hparams.lr if hasattr(self.hparams, 'lr') and self.hparams.lr is not None else CONFIG['learning_rate'],
            weight_decay=CONFIG['weight_decay']
        )
        # TODO: softcode reduce lr on plateau params
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val/loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

model = LitModel(
    lr=CONFIG['learning_rate'],
    n_classes=dm.n_classes
)
model

Using cache found in /home/tsilva/.cache/torch/hub/pytorch_vision_main


LitModel(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runn

Find optimal batch size:

In [12]:
#from pytorch_lightning.tuner import Tuner
#trainer = pl.Trainer()
#tuner = Tuner(trainer)
#tuner.scale_batch_size(model, datamodule=dm, mode="power")

First try to overfit a batch to make sure the training pipeline works:

In [13]:
class ThresholdStoppingCallback(pl.Callback):
    def __init__(self, metric, threshold):
        super().__init__()
        self.metric = metric
        self.threshold = threshold
        
    def on_train_epoch_end(self, trainer, pl_module):
        value = trainer.callback_metrics.get(self.metric)
        if value >= self.threshold:
            print(f"Stopping training as {self.metric} reached {self.threshold}")
            trainer.should_stop = True

trainer = pl.Trainer(
    log_every_n_steps=1,
    overfit_batches=1,
    max_epochs=100,
    callbacks=[ThresholdStoppingCallback("train/acc", 1.0)]
)
model = LitModel(
    lr=CONFIG['learning_rate'],
    n_classes=dm.n_classes
)
trainer.fit(model, datamodule=create_data_module(CONFIG['dataset'], batch_size=32, train_shuffle=False))

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
`Trainer(overfit_batches=1)` was configured so 1 batch will be used.
Using cache found in /home/tsilva/.cache/torch/hub/pytorch_vision_main


Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | ResNet           | 11.2 M | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.727    Total estimated model params size (MB)
69        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Stopping training as train/acc reached 1.0


Run the training loop:

In [ ]:
import time
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import Timer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.callbacks import StochasticWeightAveraging

class EpochTimeLogger(pl.Callback):
    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start_time = time.time()

    def on_train_epoch_end(self, trainer, pl_module):
        epoch_time = time.time() - self.epoch_start_time
        if trainer.logger is not None and hasattr(trainer.logger, "experiment"):
            trainer.logger.experiment.log({"epoch_time_sec": epoch_time, "epoch": trainer.current_epoch})

import pytorch_lightning as pl
from typing import Union

# TODO: print model summary
class BackboneWarmupCallback(pl.Callback):
    def __init__(self, unfreeze_at: Union[float, int]):
        if isinstance(unfreeze_at, float):
            assert 0.0 < unfreeze_at < 1.0, "Percentage must be between 0 and 1."
        elif isinstance(unfreeze_at, int):
            assert unfreeze_at >= 0, "Epoch number must be non-negative."
        else:
            raise TypeError("`unfreeze_at` must be a float (percentage) or an int (epoch number).")
        
        self.unfreeze_at = unfreeze_at
        self.unfrozen = False
        self.unfreeze_epoch = None

    def on_train_start(self, trainer, pl_module):
        if isinstance(self.unfreeze_at, float):
            self.unfreeze_epoch = int(self.unfreeze_at * trainer.max_epochs)
        else:
            self.unfreeze_epoch = self.unfreeze_at

    def on_train_epoch_start(self, trainer, pl_module):
        if self.unfrozen:
            return

        if trainer.current_epoch == 0:
            print(f"[Epoch {trainer.current_epoch}] Training with frozen backbone until epoch {self.unfreeze_epoch}...")
            pl_module.freeze_backbone()
        elif trainer.current_epoch >= self.unfreeze_epoch:
            print(f"[Epoch {trainer.current_epoch}] Unfroze backbone... now training all layers.")
            pl_module.unfreeze_backbone()
            self.unfrozen = True

trainer_callbacks = [
    Timer(), 
    EpochTimeLogger(), 
    ModelCheckpoint(
        monitor="val/loss",
        save_top_k=1,
        mode="min",
        save_last=True,
        dirpath="temp/checkpoints",
        filename=os.environ["NOTEBOOK_ID"] + "-{epoch:02d}-{val/loss:.2f}"
    ), 
    LearningRateMonitor(logging_interval="epoch"),
    EarlyStopping(
        monitor="val/loss",
        patience=CONFIG['early_stopping_patience'],
        min_delta=CONFIG['early_stopping_min_delta'],
        mode="min",
        verbose=True,
        strict=True
    ) if CONFIG['early_stopping_patience'] > 0 else None,
    StochasticWeightAveraging(swa_lrs=CONFIG['swa_lrs']) if CONFIG['swa_lrs'] > 0 else None,
    BackboneWarmupCallback(CONFIG['backbone_warmup_percentage']) if CONFIG['backbone_warmup_percentage'] > 0 else None
]
trainer_callbacks = [cb for cb in trainer_callbacks if cb is not None]

num_gpus = torch.cuda.device_count()
trainer = pl.Trainer(
    devices=num_gpus,
    accelerator="auto",
    strategy="auto",
    benchmark=True,
    max_epochs=CONFIG['n_epochs'],
    log_every_n_steps=5,
    logger=WandbLogger(project=os.environ["NOTEBOOK_ID"], config=CONFIG),
    callbacks=trainer_callbacks,
    precision="16-mixed" if CONFIG['use_mixed_precision'] else 32
)
model = LitModel(
    lr=CONFIG['learning_rate'],
    n_classes=dm.n_classes
)
trainer.fit(model, datamodule=dm)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Using cache found in /home/tsilva/.cache/torch/hub/pytorch_vision_main


Files already downloaded and verified
Files already downloaded and verified


/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/tsilva/repos/tsilva/aiml-notebooks/notebooks/temp/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | ResNet           | 11.2 M | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.727    Total estimated model params size (MB)
69        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val/loss improved by 0.302 >= min_delta = 0.0001. New best score: 0.327


Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Calculate the post-training test set accuracy:

In [ ]:
single_gpu_trainer = pl.Trainer(
    devices=1,
    accelerator="auto"
)
single_gpu_trainer.test(model, datamodule=dm)